In [1]:
%load_ext autoreload
%autoreload 2
import os
import matplotlib.pyplot as plt
import seaborn as sns
from os.path import join
from tqdm import tqdm
import pandas as pd
import sys
import joblib
from scipy.special import softmax
from neuro import config
import numpy as np
from collections import defaultdict
from copy import deepcopy
import pandas as pd
import neuro.sasc.viz
import neuro.viz
from neuro.sasc import analyze_helper
import neuro.sasc
from neuro.sasc.modules.fmri_module import convert_module_num_to_voxel_num
from scipy.stats import false_discovery_control
import dvu
dvu.set_style()

# pcs = joblib.load(join(FMRI_DIR, "voxel_neighbors_and_pcs", "loo_pc_UTS02.pkl"))
# pcs['good_voxels'].shape
# pcs['pca_projections'].shape

<frozen importlib._bootstrap>:488: DeprecationWarning: builtin type SwigPyPacked has no __module__ attribute
<frozen importlib._bootstrap>:488: DeprecationWarning: builtin type SwigPyObject has no __module__ attribute
/home/chansingh/automated-brain-explanations/.venv/lib/python3.12/site-packages/spacy/cli/_util.py:23: DeprecationWarning: Importing 'parser.split_arg_string' is deprecated, it will only be available in 'shell_completion' in Click 9.0.
  from click.parser import split_arg_string
/home/chansingh/automated-brain-explanations/.venv/lib/python3.12/site-packages/weasel/util/config.py:8: DeprecationWarning: Importing 'parser.split_arg_string' is deprecated, it will only be available in 'shell_completion' in Click 9.0.
  from click.parser import split_arg_string


In [2]:
pilot_name = 'pilot_story_data.pkl'
# pilot_name = 'pilot3_story_data.pkl'
# pilot_name = 'pilot4_story_data.pkl'
# pilot_name = "pilot6_story_data.pkl"
# pilot_name = "pilot8_story_data.pkl"

stories_data_dict = joblib.load(
    join(config.RESULTS_DIR_LOCAL, 'gct_processed', pilot_name))
if pilot_name == 'pilot_story_data.pkl':
    pilot_data_dir = join(config.PILOT_STORY_DATA_DIR, '20230504')
# elif pilot_name == 'pilot3_story_data.pkl':
#     pilot_data_dir = join(config.PILOT_STORY_DATA_DIR, '20231106')
# elif pilot_name == 'pilot4_story_data.pkl':
#     pilot_data_dir = join(config.PILOT_STORY_DATA_DIR, '20240509')
# elif pilot_name == 'pilot6_story_data.pkl':
#     pilot_data_dir = join(config.PILOT_STORY_DATA_DIR, '20241202')
# elif pilot_name == 'pilot7_story_data.pkl':
#     pilot_data_dir = join(config.PILOT_STORY_DATA_DIR, '20241204')
# elif pilot_name == 'pilot8_story_data.pkl':
#     pilot_data_dir = join(config.PILOT_STORY_DATA_DIR, '20241204')

In [3]:
# load responses
default_story_idxs = np.array([0, 1, 2, 3, 5])
# np.where(
    # (np.array(stories_data_dict['story_setting']) == 'default') |
    # (np.array(stories_data_dict['story_setting']) == 'roi')
# )[0]
resp_np_files = [stories_data_dict['story_name_new'][i].replace('_resps', '')
                 for i in default_story_idxs]
resps_dict = {
    k: np.load(join(pilot_data_dir, k))
    for k in tqdm(resp_np_files)
}

  0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 5/5 [00:08<00:00,  1.79s/it]


In [10]:
mats = defaultdict(list)
resp_chunks = defaultdict(list)
use_clusters_list = [False]
for use_clusters in use_clusters_list:
    for story_num in default_story_idxs:
        rows = stories_data_dict["rows"][story_num]

        # get resp_chunks
        resp_story = resps_dict[
            stories_data_dict["story_name_new"][story_num].replace(
                '_resps', '')
        ].T  # (voxels, time)
        timing = stories_data_dict["timing"][story_num]
        if 'paragraphs' in stories_data_dict.keys():
            paragraphs = stories_data_dict["paragraphs"][story_num]
        else:
            paragraphs = stories_data_dict["story_text"][story_num].split(
                "\n\n")
        # paragraphs = stories_data_dict["story_text"][story_num].split("\n\n")
        if pilot_name in ['pilot3_story_data.pkl']:
            paragraphs = [neuro.sasc.analyze_helper.remove_repeated_words(
                p) for p in paragraphs]
        assert len(paragraphs) == len(
            rows), f"{len(paragraphs)} != {len(rows)}"
        resp_chunks = analyze_helper.get_resps_for_paragraphs(
            timing, paragraphs, resp_story, offset=2, validate=True,
            split_hyphens=pilot_name in ["pilot6_story_data.pkl", "pilot7_story_data.pkl", "pilot8_story_data.pkl"])
        assert len(resp_chunks) <= len(paragraphs)

        # calculate mat
        mat = np.zeros((len(rows), len(paragraphs)))
        for i in range(len(resp_chunks)):
            if use_clusters == False:
                # driving single voxel
                if 'voxel_num' in rows.columns:
                    mat[:, i] = resp_chunks[i][rows["voxel_num"].values].mean(
                        axis=1).flatten()
                elif 'voxel_nums' in rows.columns:
                    mat[:, i] = [resp_chunks[i][x].mean()
                                 for x in rows['voxel_nums']]
                    # resp_chunks[i][rows["voxel_nums"].values].mean(
                    # axis=1).flatten()

            elif use_clusters == True:
                for r in range(len(rows)):
                    cluster_nums = rows.iloc[r]["cluster_nums"]
                    if isinstance(cluster_nums, np.ndarray):
                        vals = resp_chunks[i][cluster_nums].flatten()
                        mat[r, i] = np.nanmean(vals)
                    else:
                        # print(cluster_nums)
                        mat[r, i] = np.nan
        mat[:, 0] = np.nan  # ignore the first column
        # print('mat', mat)

        # sort by voxel_num
        if 'voxel_num' in rows.columns:
            args = np.argsort(rows["voxel_num"].values)
        elif pilot_name in ["pilot6_story_data.pkl", "pilot7_story_data.pkl", "pilot8_story_data.pkl"]:
            args = np.argsort(rows["expl"].values)
        else:
            args = np.argsort(rows["roi"].values)
        mat = mat[args, :][:, args]
        mats[use_clusters].append(deepcopy(mat))

        # plt.imshow(mat)
        # plt.colorbar(label="Mean response")
        # plt.xlabel("Corresponding paragraph\n(Ideally, diagonal should be brighter)")
        # plt.ylabel("Voxel")
        # plt.title(f"{story_data['story_name_new'][story_num][3:-10]}")
        # plt.show()

if 'voxel_num' in rows.columns:
    rows = rows.sort_values(by="voxel_num")
elif pilot_name in ["pilot6_story_data.pkl", "pilot7_story_data.pkl", "pilot8_story_data.pkl"]:
    rows = rows.sort_values(by="expl")
else:
    rows = rows.sort_values(by="roi")
expls = rows["expl"].values


m = {}
for use_clusters in use_clusters_list:
    mats[use_clusters] = np.array(mats[use_clusters])  # (6, 17, 17)
    m[use_clusters] = np.nanmean(mats[use_clusters], axis=0)
    m['std'] = np.nanstd(mats[use_clusters], axis=0)

In [11]:
m

{False: array([[-0.11289183,  0.32269777,  0.37348175,  0.25664591, -0.36380242,
          0.25747366, -0.32631107,  0.02137578,  0.26105519,  0.06429627,
         -0.0506615 ,  0.20792304, -0.1606421 ,  0.20589309, -0.21085247,
         -0.45134074, -0.46711821],
        [-0.27511998, -0.09919275,  0.18557712,  0.33871935, -0.30853915,
          0.40375489, -0.12602108,  0.06313212,  0.25011005,  0.10038821,
          0.10016235,  0.09820077, -0.22364916,  0.16326122, -0.30004679,
         -0.46193169, -0.31726306],
        [-0.03793384,  0.02064044,  0.12604308,  0.20510513, -0.1139451 ,
          0.23109474, -0.16867243,  0.08080592, -0.01596453, -0.12304382,
         -0.23634261,  0.00898694, -0.02959127,  0.01913424, -0.11793598,
         -0.40514296, -0.12537784],
        [-0.22473867, -0.2712578 ,  0.15649574,  0.1736287 , -0.04963124,
          0.17715557, -0.10704991,  0.07543162,  0.13834188,  0.00085937,
          0.19345456,  0.40340234, -0.10305239,  0.42298539, -0.0719351

In [12]:
# make a column of tuples of (expl, voxel_num)
col = [(expl, vn) for expl, vn in zip(rows['expl'], rows['voxel_num'])]

In [13]:
means = pd.DataFrame(m[False], columns=col, index=col)
means.index.name = 'Driving paragraph (expl, voxel_num)'
means.columns.name = 'Voxel to be driven (expl, voxel_num)'
stds = pd.DataFrame(m['std'], columns=col, index=col)
stds.index.name = 'Driving paragraph (expl, voxel_num)'
stds.columns.name = 'Voxel to be driven (expl, voxel_num)'
joblib.dump({'means': means, 'stds': stds}, 'means_and_stds_5stories_audio_comparison.joblib')

['means_and_stds_5stories_audio_comparison.joblib']